<a href="https://colab.research.google.com/github/ishreya-dev/tinystories-gpt/blob/v1-char-level/slm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install datasets

In [2]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

True
Tesla T4


In [3]:
from datasets import load_dataset

dataset = load_dataset("roneneldan/TinyStories")

README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…): reconstructing file:   0%|          |  0.00B /  249MB            

data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-5852b56a2bd28f(…): reconstructing file:   0%|          |  0.00B /  248MB            

data/train-00001-of-00004-5852b56a2bd28f(…): downloading bytes:           |  0.00B            

...

data/train-00003-of-00004-d243063613e5a0(…): downloading bytes:           |  0.00B            

data/validation-00000-of-00001-869c898b5(…): reconstructing file:   0%|          |  0.00B / 9.99MB            

data/validation-00000-of-00001-869c898b5(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [4]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})


In [5]:
print(dataset["train"][0])

{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}


In [6]:
dataset["train"][0]["text"]

'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'

In [7]:
print("Number of training examples:", len(dataset["train"]))
print("Number of validation examples:", len(dataset["validation"]))

print("\nColumns:")
print(dataset["train"].column_names)

print("\nFirst story:")
print(dataset["train"][0]["text"])

Number of training examples: 2119719
Number of validation examples: 21990

Columns:
['text']

First story:
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.


In [8]:
for i in range(3):
  print(f"\n--- Story {i+1} ---")
  print(dataset["train"][i]["text"][:300])


--- Story 1 ---
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and 

--- Story 2 ---
Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.

One day, Beep was driving in the park when he saw a big tree. The tree had many leaves that were falling. Bee

--- Story 3 ---
One day, a little fish named Fin was swimming near the shore. He saw a big crab and wanted to be friends. "Hi, I am Fin. Do you want to play?" asked the little fish. The crab looked at Fin and said, "No, I don't want to play. I am cold and I don't feel fine."

Fin felt sad but wanted to help the cra


In [9]:
# Take a small portion of the training data first
text = "\n".join(dataset["train"][:1000]["text"])

print(text[:500])
print("\nTotal characters:", len(text))

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them b

Total characters: 942639


In [10]:
# Find all unique characters
chars = sorted(list(set(text)))

# Vocabulary size
vocab_size = len(chars)

print("Vocabulary size:", vocab_size)
print(chars)

Vocabulary size: 75
['\n', ' ', '!', '"', '$', "'", ',', '-', '.', '0', '1', '2', '3', '8', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'â', 'œ', '“', '”', '€', '™']


In [11]:
# String → Integer
stoi = {ch: i for i, ch in enumerate(chars)}

# Integer → String
itos = {i: ch for i, ch in enumerate(chars)}

print(stoi)

{'\n': 0, ' ': 1, '!': 2, '"': 3, '$': 4, "'": 5, ',': 6, '-': 7, '.': 8, '0': 9, '1': 10, '2': 11, '3': 12, '8': 13, ':': 14, ';': 15, '?': 16, 'A': 17, 'B': 18, 'C': 19, 'D': 20, 'E': 21, 'F': 22, 'G': 23, 'H': 24, 'I': 25, 'J': 26, 'K': 27, 'L': 28, 'M': 29, 'N': 30, 'O': 31, 'P': 32, 'Q': 33, 'R': 34, 'S': 35, 'T': 36, 'U': 37, 'V': 38, 'W': 39, 'X': 40, 'Y': 41, 'Z': 42, 'a': 43, 'b': 44, 'c': 45, 'd': 46, 'e': 47, 'f': 48, 'g': 49, 'h': 50, 'i': 51, 'j': 52, 'k': 53, 'l': 54, 'm': 55, 'n': 56, 'o': 57, 'p': 58, 'q': 59, 'r': 60, 's': 61, 't': 62, 'u': 63, 'v': 64, 'w': 65, 'x': 66, 'y': 67, 'z': 68, 'â': 69, 'œ': 70, '“': 71, '”': 72, '€': 73, '™': 74}


Character level tokenizser

In [12]:
def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

In [13]:
example = "Hello Lily!"

encoded = encode(example)

print("Original:", example)
print("Encoded:", encoded)
print("Decoded:", decode(encoded))

Original: Hello Lily!
Encoded: [24, 47, 54, 54, 57, 1, 28, 51, 54, 67, 2]
Decoded: Hello Lily!


In [14]:
# Convert our entire text into token IDs
data = encode(text)

print(data[:20])
print("Total tokens:", len(data))

[31, 56, 47, 1, 46, 43, 67, 6, 1, 43, 1, 54, 51, 62, 62, 54, 47, 1, 49, 51]
Total tokens: 942639


In [15]:
example_data = data[:20]

x = example_data[:-1]
y = example_data[1:]

print("Input IDs: ", x)
print("Target IDs:", y)

print("\nInput text: ", decode(x))
print("Target text:", decode(y))

Input IDs:  [31, 56, 47, 1, 46, 43, 67, 6, 1, 43, 1, 54, 51, 62, 62, 54, 47, 1, 49]
Target IDs: [56, 47, 1, 46, 43, 67, 6, 1, 43, 1, 54, 51, 62, 62, 54, 47, 1, 49, 51]

Input text:  One day, a little g
Target text: ne day, a little gi


In [16]:
import torch
import torch.nn as nn
from torch.nn import functional as F

data = torch.tensor(encode(text), dtype=torch.long)

print(data[:20])
print(type(data))
print(data.dtype)

tensor([31, 56, 47,  1, 46, 43, 67,  6,  1, 43,  1, 54, 51, 62, 62, 54, 47,  1,
        49, 51])
<class 'torch.Tensor'>
torch.int64


In [17]:
# 90% for training, 10% for validation

n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

print("Training tokens:", len(train_data))
print("Validation tokens:", len(val_data))

Training tokens: 848375
Validation tokens: 94264


In [18]:
batch_size = 32
block_size = 64

n_embd = 128      # size of each embedding vector
n_head = 4        # number of attention heads
n_layer = 4       # number of Transformer blocks

dropout = 0.1

device = "cuda" if torch.cuda.is_available() else "cpu"

In [19]:
token_embedding_table = nn.Embedding(
    vocab_size,
    n_embd
).to(device)

In [20]:
def get_batch(split):

    # Select train or validation data
    data_split = train_data if split == "train" else val_data

    # Random starting positions
    ix = torch.randint(
        len(data_split) - block_size,
        (batch_size,)
    )

    # Input sequences
    x = torch.stack([
        data_split[i:i + block_size]
        for i in ix
    ])

    # Target sequences
    y = torch.stack([
        data_split[i + 1:i + block_size + 1]
        for i in ix
    ])

    return x.to(device), y.to(device)

In [21]:
xb, yb = get_batch("train")

token_embeddings = token_embedding_table(xb)

print("Input shape:", xb.shape)
print("Target shape:", yb.shape)
print("Embedding shape:", token_embeddings.shape)

Input shape: torch.Size([32, 64])
Target shape: torch.Size([32, 64])
Embedding shape: torch.Size([32, 64, 128])


In [22]:
print(decode(xb[0].tolist()))

 Mia, don't give up," Tim says. "We are almost there. I want to 


In [23]:
print("INPUT:")
print(decode(xb[0].tolist()))

print("\nTARGET:")
print(decode(yb[0].tolist()))

INPUT:
 Mia, don't give up," Tim says. "We are almost there. I want to 

TARGET:
Mia, don't give up," Tim says. "We are almost there. I want to s


In [24]:
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(
            vocab_size,
            vocab_size
        )

    def forward(self, idx, targets=None):

        logits = self.token_embedding_table(idx)

        loss = None

        if targets is not None:

            B, T, C = logits.shape

            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            # Get predictions
            logits, loss = self(idx)

            # Take prediction for the last character
            logits = logits[:, -1, :]

            # Convert scores into probabilities
            probs = F.softmax(logits, dim=-1)

            # Sample the next character
            idx_next = torch.multinomial(
                probs,
                num_samples=1
            )

            # Add the new character
            idx = torch.cat(
                (idx, idx_next),
                dim=1
            )

        return idx

In [25]:
model = BigramLanguageModel(vocab_size)
model = model.to(device)

logits, loss = model(xb, yb)

print("Logits shape:", logits.shape)
print("Loss:", loss)

Logits shape: torch.Size([2048, 75])
Loss: tensor(4.7111, device='cuda:0', grad_fn=<NllLossBackward0>)


In [26]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3
)

In [27]:
max_iters = 5000

for step in range(max_iters):

    # Get a random batch
    xb, yb = get_batch("train")

    # Forward pass
    logits, loss = model(xb, yb)

    # Remove gradients from the previous step
    optimizer.zero_grad()

    # Calculate gradients
    loss.backward()

    # Update model parameters
    optimizer.step()

    # Print loss every 500 steps
    if step % 500 == 0:
        print(f"Step {step}: Loss = {loss.item():.4f}")

Step 0: Loss = 4.7141
Step 500: Loss = 4.0529
Step 1000: Loss = 3.4597
Step 1500: Loss = 3.0764
Step 2000: Loss = 2.8087
Step 2500: Loss = 2.6554
Step 3000: Loss = 2.5426
Step 3500: Loss = 2.4713
Step 4000: Loss = 2.3913
Step 4500: Loss = 2.4052


In [28]:
def generate(self, idx, max_new_tokens):

    for _ in range(max_new_tokens):

        # Get predictions
        logits, loss = self(idx)

        # Take logits from the last position
        logits = logits[:, -1, :]

        # Convert logits into probabilities
        probs = F.softmax(logits, dim=-1)

        # Sample the next token
        idx_next = torch.multinomial(probs, num_samples=1)

        # Add the predicted token
        idx = torch.cat((idx, idx_next), dim=1)

    return idx

In [29]:
# Start with a newline character
context = torch.zeros((1, 1), dtype=torch.long, device=device)

generated_ids = model.generate(
    context,
    max_new_tokens=500
)

generated_text = decode(
    generated_ids[0].tolist()
)

print(generated_text)




On d 



Ond TiAnd hon dddand kirl t g. savedenghe y ad t a c$Hrtpoug, hatoked, hQfugixcawhud. andd t, dthul thep. whed.
"Shed mHe as. hey ollicite " cin he h. bene wQ't aigin trTmMar llire!Bume The o ht ary howLiczie Toomlle, fare ithishe Sp caknit”El She ane.goxche! ucauWhw lith.
On, lo wa2Whe Sh.âCanczee marry.ntitoy "er he k E, Tol-cheYoromilas, sQmayoune ca bethedain me w aK?â€”“Wheredppe t. g seBes a salotalo XThoour wariJolEs le o ors waP?", sainnd oulyw y lo toxy,œse and somey. theve b


In [30]:
# Positional embedding table
position_embedding_table = nn.Embedding(
    block_size,
    n_embd
).to(device)

# Positions: 0, 1, 2, ..., 63
positions = torch.arange(block_size, device=device)

# Get embedding for each position
position_embeddings = position_embedding_table(positions)

print("Position shape:", positions.shape)
print("Position embedding shape:", position_embeddings.shape)

Position shape: torch.Size([64])
Position embedding shape: torch.Size([64, 128])


In [31]:
x = token_embeddings + position_embeddings

print("Final input shape:", x.shape)

Final input shape: torch.Size([32, 64, 128])


In [32]:
class Head(nn.Module):
    """One head of self-attention"""
    def __init__(self, head_size):
        self.head_size = head_size
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer(
            "tril",
            torch.tril(torch.ones(block_size, block_size))
            )

    def forward(self, x):

        B, T, C = x.shape

        # Create Key, Query, and Value
        k = self.key(x)      # (B, T, head_size)
        q = self.query(x)    # (B, T, head_size)
        v = self.value(x)    # (B, T, head_size)

        # Calculate attention scores
        wei = q @ k.transpose(-2, -1)

        # Scale attention scores
        wei = wei * self.head_size ** -0.5

        # Mask future positions
        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
            )

        # Convert attention scores into probabilities
        wei = F.softmax(wei, dim=-1)

        # Use attention probabilities to combine the Value vectors
        out = wei @ v

        return out

In [33]:
head = Head(head_size=32).to(device)

attention_scores = head(x)

print(attention_scores.shape)

torch.Size([32, 64, 32])


In [34]:
head = Head(head_size=32).to(device)

out = head(x)

print("Output shape:", out.shape)

Output shape: torch.Size([32, 64, 32])


In [35]:
class MultiHeadAttention(nn.Module):
    """Multiple attention heads running in parallel"""

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList(
            [Head(head_size) for _ in range(num_heads)]
        )

    def forward(self, x):

        # Run every attention head
        outputs = [head(x) for head in self.heads]

        # Concatenate outputs
        out = torch.cat(outputs, dim=-1)

        return out

In [36]:
head_size = n_embd // n_head

multi_head = MultiHeadAttention(
    num_heads=n_head,
    head_size=head_size
).to(device)

out = multi_head(x)

print("Output shape:", out.shape)

Output shape: torch.Size([32, 64, 128])


In [37]:
class MultiHeadAttention(nn.Module):
    """Multiple attention heads running in parallel"""

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList(
            [Head(head_size) for _ in range(num_heads)]
        )

        # Project combined attention output
        self.proj = nn.Linear(
            num_heads * head_size,
            n_embd
        )

    def forward(self, x):

        # Run all attention heads
        out = torch.cat(
            [head(x) for head in self.heads],
            dim=-1
        )

        # Output projection
        out = self.proj(out)

        return out

In [38]:
multi_head = MultiHeadAttention(
    num_heads=n_head,
    head_size=n_embd // n_head
).to(device)

out = multi_head(x)

print("Output shape:", out.shape)

Output shape: torch.Size([32, 64, 128])


In [39]:
class FeedForward(nn.Module):
    """A simple feed-forward network"""

    def __init__(self, n_embd):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd)
        )

    def forward(self, x):
        return self.net(x)

In [40]:
ffwd = FeedForward(n_embd).to(device)

out = ffwd(x)

print("Output shape:", out.shape)

Output shape: torch.Size([32, 64, 128])


In [41]:
class Block(nn.Module):
    """Transformer block"""

    def __init__(self, n_embd, n_head):
        super().__init__()

        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(
            n_head,
            head_size
        )

        self.ffwd = FeedForward(n_embd)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):

        # Self-attention + residual connection
        x = x + self.sa(self.ln1(x))

        # Feed-forward + residual connection
        x = x + self.ffwd(self.ln2(x))

        return x

In [42]:
block = Block(
    n_embd=n_embd,
    n_head=n_head
).to(device)

out = block(x)

print("Input shape:", x.shape)
print("Output shape:", out.shape)

Input shape: torch.Size([32, 64, 128])
Output shape: torch.Size([32, 64, 128])


In [43]:
class TransformerLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        # Token embeddings
        self.token_embedding_table = nn.Embedding(
            vocab_size,
            n_embd
        )

        # Position embeddings
        self.position_embedding_table = nn.Embedding(
            block_size,
            n_embd
        )

        # Stack multiple Transformer blocks
        self.blocks = nn.Sequential(
            *[
                Block(
                    n_embd=n_embd,
                    n_head=n_head
                )
                for _ in range(n_layer)
            ]
        )

        # Final layer normalization
        self.ln_f = nn.LayerNorm(n_embd)

        # Convert embeddings to vocabulary logits
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size
        )

    def forward(self, idx, targets=None):

        B, T = idx.shape

        # Token embeddings
        tok_emb = self.token_embedding_table(idx)

        # Position embeddings
        pos_emb = self.position_embedding_table(
            torch.arange(T, device=idx.device)
        )

        # Combine token and position information
        x = tok_emb + pos_emb

        # Pass through Transformer blocks
        x = self.blocks(x)

        # Final normalization
        x = self.ln_f(x)

        # Convert to vocabulary predictions
        logits = self.lm_head(x)

        loss = None

        if targets is not None:

            B, T, C = logits.shape

            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=1.0):
      for _ in range(max_new_tokens):

        # Keep only the last block_size characters
        idx_cond = idx[:, -block_size:]

        # Forward pass
        logits, _ = self(idx_cond)

        # Take only the last character prediction
        logits = logits[:, -1, :]

        # Apply temperature
        logits = logits / temperature

        # Convert to probabilities
        probs = F.softmax(logits, dim=-1)

        # Sample next character
        idx_next = torch.multinomial(
            probs,
            num_samples=1
        )

        # Add generated character
        idx = torch.cat(
            (idx, idx_next),
            dim=1
        )
      return idx

In [44]:
model = TransformerLanguageModel().to(device)

xb, yb = get_batch("train")

logits, loss = model(xb, yb)

print("Logits shape:", logits.shape)
print("Loss:", loss)

Logits shape: torch.Size([2048, 75])
Loss: tensor(4.5100, device='cuda:0', grad_fn=<NllLossBackward0>)


In [45]:
model = TransformerLanguageModel().to(device)

xb, yb = get_batch("train")

logits, loss = model(xb, yb)

print("Logits shape:", logits.shape)
print("Loss:", loss)

Logits shape: torch.Size([2048, 75])
Loss: tensor(4.4521, device='cuda:0', grad_fn=<NllLossBackward0>)


In [46]:
@torch.no_grad()
def estimate_loss():
    out = {}

    model.eval()

    for split in ["train", "val"]:
        losses = torch.zeros(100)

        for k in range(100):
            X, Y = get_batch(split)

            logits, loss = model(X, Y)

            losses[k] = loss.item()

        out[split] = losses.mean()

    model.train()

    return out

In [47]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

In [48]:
max_iters = 5000
eval_interval = 500

for step in range(max_iters):

    # Evaluate train and validation loss
    if step % eval_interval == 0:
        losses = estimate_loss()

        print(
            f"Step {step}: "
            f"Train Loss = {losses['train']:.4f}, "
            f"Val Loss = {losses['val']:.4f}"
        )

    # Get training batch
    xb, yb = get_batch("train")

    # Forward pass
    logits, loss = model(xb, yb)

    # Clear old gradients
    optimizer.zero_grad()

    # Calculate gradients
    loss.backward()

    # Update model weights
    optimizer.step()

Step 0: Train Loss = 4.4836, Val Loss = 4.4796
Step 500: Train Loss = 2.0508, Val Loss = 2.0283
Step 1000: Train Loss = 1.6530, Val Loss = 1.6776
Step 1500: Train Loss = 1.4805, Val Loss = 1.5002
Step 2000: Train Loss = 1.3740, Val Loss = 1.4157
Step 2500: Train Loss = 1.3073, Val Loss = 1.3619
Step 3000: Train Loss = 1.2581, Val Loss = 1.3186
Step 3500: Train Loss = 1.2246, Val Loss = 1.2864
Step 4000: Train Loss = 1.1827, Val Loss = 1.2619
Step 4500: Train Loss = 1.1515, Val Loss = 1.2335


In [49]:
def generate(self, idx, max_new_tokens):

    for _ in range(max_new_tokens):

        # Keep only the last block_size tokens
        idx_cond = idx[:, -block_size:]

        # Get predictions
        logits, loss = self(idx_cond)

        # Take logits from the last position
        logits = logits[:, -1, :]

        # Convert logits to probabilities
        probs = F.softmax(logits, dim=-1)

        # Sample the next token
        idx_next = torch.multinomial(
            probs,
            num_samples=1
        )

        # Append the new token
        idx = torch.cat(
            (idx, idx_next),
            dim=1
        )

    return idx

In [50]:
context = torch.zeros(
    (1, 1),
    dtype=torch.long,
    device=device
)

generated_ids = model.generate(
    context,
    max_new_tokens=1000,
    temperature=0.7
)

generated_text = decode(
    generated_ids[0].tolist())

print(generated_text)


Once upon a time the felt saw a pig on came on the water and scared special from ones. They was so to see so many to star to see a people in the hole. They wanted to their adventures and to play with his friend, a boy to get believe remembermored out to joy. She was so happy and see his toys and always happy. They felt said it seem lonely and it soon the toy sailed and stopped it up. They had some one and he liked to be hars. They has a boy. They were happy and the park when the princes and was so explain and he liked that couldn't was so excared. She saw a time there was a little girl. He loved to see what so a gone. They had come our showeed to see boy the stone.

She liked to get to help his but of his dad. She kept melt played to his arm and the ground and so happy. She ran on the surprises had and explored with a bug! 

The rain, "Anna. It can Sam safe you, I want to be noise. I like a broken some come with in the mine. It is promiss up in. She wanted to come to a nap. They want 